In [ ]:
import os
import h5py
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import mplhep as hep
from tqdm.notebook import tqdm
import hist
from ipywidgets import interact

from wremnants.utilities.io_tools import base_io
from wremnants.postprocessing.datagroups.datagroups import Datagroups
from wums import ioutils, logging
from wums import boostHistHelpers as hh

import ROOT
from ROOT import RooFit, RooRealVar, RooArgList, RooFormulaVar
from ROOT import kRed, kBlue, kGreen, kMagenta

import random
import string

%jsroot on

# Enable multi-threading
ROOT.ROOT.EnableImplicitMT()

plt.figure() # Plotting an empty figure fixes incorrect style loading ?!'
plt.close()
plt.style.use(hep.style.CMS)
plt.rcParams["figure.facecolor"] = 'white' # needed to remove transparent background

In [ ]:
def SetOwnerships(root_obj):
    """
    Gives pyROOT the ability to close the given ROOT objects when they become unused.
    Reclaiming all defined ROOT objects slows down the memory leakage.
    """
    if type(root_obj) is list:
        for i in range(len(root_obj)):
            ROOT.SetOwnership(root_obj[i], True)

    else:
        ROOT.SetOwnership(root_obj, True)

def make_root_hist(hist, hist_name="hist"):
    hist_TH1D = ROOT.TH1D(hist_name, "hist", len(hist.axes[0]), hist.axes[0].edges)
    for i in range(hist.axes[0].size):
        hist_TH1D.SetBinContent(i+1, hist[i].value)
        hist_TH1D.SetBinError(i+1, hist[i].variance**0.5)
    
    return hist_TH1D

def get_chi2_for_bin(hist1, hist2, model1, model2, x_var, hist1_name, hist2_name):
    rooHist1 = ROOT.RooDataHist("hist1_name", "dh", [x_var], hist1)
    rooHist2 = ROOT.RooDataHist("hist2_name", "dh", [x_var], hist2)
    
    test_chi2_1 = model1.createChi2(rooHist1, DataError="Expected", Verbose=True).getVal()
    test_chi2_2 = model2.createChi2(rooHist2, DataError="Expected", Verbose=True).getVal()

    return test_chi2_1 + test_chi2_2

def plot_bias_distributions(bias_arr, bias_err_arr, bin_centers_i, bin_centers_j,
                            save_fig=False,
                            save_paths=None,
                            title=None,
                            unit_label=None,
                            is_data=True,
                            bias_amplitude=None,
                            bias_err_amplitude=None,
                            bias_err_ratio_amplitude=None,
                            extend_colorbars=[None, None]):  

    bias_arr     = bias_arr.copy()
    bias_err_arr = bias_err_arr.copy()
    bias_err_arr[np.isclose(bias_err_arr, 0)] = np.nan
    if bias_amplitude is None:
        bias_amplitude = np.nanmax(np.abs(bias_arr))
    if bias_err_amplitude is None:
        bias_err_amplitude = np.nanmax(np.abs(bias_err_arr))
    if bias_err_ratio_amplitude is None:
        bias_err_ratio_amplitude = np.nanmax(np.abs(bias_arr/bias_err_arr))
    
    fig, ax = plt.subplots(3, 1, figsize=(11,18))
    hep.cms.label(loc=0, rlabel="", data=is_data, label="Preliminary", ax=ax[0])

    fig.suptitle(title)

    cmap1 = plt.get_cmap("bwr").copy()
    cmap1.set_extremes(under='magenta', over='yellow')
    cmap1.set_bad('black',1.)
    im1 = hep.hist2dplot((bias_arr, bin_centers_i, bin_centers_j), ax=ax[0], cmap=cmap1)
    im1.pcolormesh.set_clim(-bias_amplitude, bias_amplitude)
    ax[0].set_title("curvature bias", pad=40)
    ax[0].set_xlabel(r"$\eta$")
    ax[0].set_ylabel(r"$\phi$")
    # ax[0].set_aspect(1)
    im1.cbar.set_label(unit_label)
    if extend_colorbars[0] != None:
        # cmap1 = im1.cbar.cmap
        # cmap1.set_extremes(under='lime', over='yellow')
        im1.cbar.remove()
        cax1 = hep.append_axes(ax[0], size="7%", pad=0.2, position="right", extend=False)
        plt.colorbar(im1.pcolormesh, ax=ax[0], cax=cax1, extend=extend_colorbars[0], label=unit_label)

    cmap2 = plt.get_cmap("magma").copy()
    cmap2.set_extremes(under='magenta', over='lime')
    im2 = hep.hist2dplot((bias_err_arr, bin_centers_i, bin_centers_j), ax=ax[1], cmap=cmap2)
    im2.pcolormesh.set_clim(0, bias_err_amplitude)
    ax[1].set_title("std_error of bias", pad=20)
    ax[1].set_xlabel(r"$\eta$")
    ax[1].set_ylabel(r"$\phi$")
    # ax[1].set_aspect(1)
    im2.cbar.set_label(unit_label)
    if extend_colorbars[1] != None:
        im2.cbar.remove()
        cax2 = hep.append_axes(ax[1], size="7%", pad=0.2, position="right", extend=False)
        plt.colorbar(im2.pcolormesh, ax=ax[1], cax=cax2, extend=extend_colorbars[1], label=unit_label)

    cmap3 = plt.get_cmap("magma").copy()
    cmap3.set_extremes(under='magenta', over='lime')
    im3 = hep.hist2dplot((np.abs(bias_arr)/bias_err_arr, bin_centers_i, bin_centers_j), ax=ax[2], cmap=cmap3)
    im3.pcolormesh.set_clim(0, bias_err_ratio_amplitude)
    ax[2].set_title("|bias| / std_error", pad=20)
    ax[2].set_xlabel(r"$\eta$")
    ax[2].set_ylabel(r"$\phi$")
    # ax[2].set_aspect(1)
    if extend_colorbars[2] != None:
        im3.cbar.remove()
        cax3 = hep.append_axes(ax[2], size="7%", pad=0.2, position="right", extend=False)
        plt.colorbar(im3.pcolormesh, ax=ax[2], cax=cax3, extend=extend_colorbars[2])
    plt.tight_layout()
    
    if save_fig:
        plt.savefig(save_paths[0], transparent=False)
        plt.clf()
    else:
        plt.show()
    
    # sigma difference between the curve fit and the bootstrap
    ratio_flattened = (bias_arr / bias_err_arr).flatten()
    ratio_flattened = ratio_flattened[~np.isnan(ratio_flattened)]
    bin_edges = np.histogram_bin_edges(ratio_flattened, bins="fd")
    x_arr = np.linspace(np.min(ratio_flattened), np.max(ratio_flattened), 1000)
    normal_dist_arr = (2*np.pi)**-0.5 * np.exp(-x_arr**2/2)
    normal_dist_arr *= len(bias_arr.flatten()) * (bin_edges[1]-bin_edges[0])
    
    plt.figure()
    hep.histplot(np.histogram((bias_arr / bias_err_arr).flatten(), density=False, bins=bin_edges), label="calculated", color="blue")
    plt.plot(x_arr, normal_dist_arr, lw=2, label="scaled normal\ndistribution\n"+r"$\sigma=1$", color="black", ls="--")
    plt.xlabel("|bias| / std_error")
    plt.ylabel("count / bin")
    if title == None:
        plt.title("Uncertainty-normalized distribution of bias", pad=20*2)
    else:
        plt.title(title + "\nUncertainty-normalized distribution of bias", pad=20*2)
    plt.legend(loc="upper right")
    hep.cms.label(loc=0, rlabel="", data=is_data, label="Preliminary")
    plt.tight_layout()
    
    if save_fig:
        plt.savefig(save_paths[1], transparent=False)
        plt.clf()
    else:
        plt.show()

def plot_bias_comparison(bias_arr_1, bias_arr_2, bin_centers_i, bin_centers_j,
                         bias_err_arr_1, bias_err_arr_2,
                         title=None,
                         title1=None, title2=None, title3=None, title4=None,
                         cbar_label_bias=None,
                         cbar_label_pull=None,
                         save_fig=False,
                         save_path=None,
                         is_data=[True, True],
                         bias_amplitude=None,
                         bias_pull_amplitude=None,
                         extend_colorbars=[None, None]):

    """
    Plots two seperate 2D curvature bias arrays, their difference and optionally their difference divided by their total uncertainty.
    """

    bias_arr_1 = bias_arr_1.copy()
    bias_arr_2 = bias_arr_2.copy()
    if bias_amplitude is None:
        bias_amplitude = np.max([np.abs(bias_arr_1), np.abs(bias_arr_2), np.abs(bias_arr_1 - bias_arr_2)])
    
    fig, ax = plt.subplots(2, 2, figsize=(18,11))
    # hep.cms.label(loc=0, rlabel="", data=is_data[0], label="Preliminary", ax=ax[0,0])

    if title is not None: fig.suptitle(title)

    ax_label_size = 35
    tilte_pad = None

    cmap1 = plt.get_cmap("bwr").copy()
    # cmap1.set_extremes(under='magenta', over='yellow')
    cmap1.set_bad('black',1.)
    im1 = hep.hist2dplot((bias_arr_1, bin_centers_i, bin_centers_j), ax=ax[0,0], cmap=cmap1)
    im1.pcolormesh.set_clim(-bias_amplitude, bias_amplitude)
    ax[0,0].set_title(title1, pad=tilte_pad)
    # ax[0,0].set_xlabel(r"$\eta$", fontsize=ax_label_size)
    ax[0,0].set_ylabel(r"$\phi$", fontsize=ax_label_size)
    im1.cbar.remove()
    # im1.cbar.set_label(cbar_label_bias)
    # if extend_colorbars[0] != None:
    #     im1.cbar.remove()
    #     cax1 = hep.append_axes(ax[0,0], size="7%", pad=0.2, position="right", extend=False)
    #     plt.colorbar(im1.pcolormesh, ax=ax[0,0], cax=cax1, extend=extend_colorbars[0], label=cbar_label_bias)
    # plt.tight_layout()
    
    cmap2 = plt.get_cmap("bwr").copy()
    # cmap2.set_extremes(under='magenta', over='yellow')
    im2 = hep.hist2dplot((bias_arr_2, bin_centers_i, bin_centers_j), ax=ax[0,1], cmap=cmap2)
    im2.pcolormesh.set_clim(-bias_amplitude, bias_amplitude)
    ax[0,1].set_title(title2, pad=tilte_pad)
    # ax[0,1].set_xlabel(r"$\eta$", fontsize=ax_label_size)
    # ax[0,1].set_ylabel(r"$\phi$", fontsize=ax_label_size)
    im2.cbar.set_label(cbar_label_bias)
    if extend_colorbars[1] != None:
        im2.cbar.remove()
        cax2 = hep.append_axes(ax[0,1], size="7%", pad=0.2, position="right", extend=False)
        plt.colorbar(im2.pcolormesh, ax=ax[0,1], cax=cax2, extend=extend_colorbars[0], label=cbar_label_bias)
    # plt.tight_layout()
    
    cmap3 = plt.get_cmap("bwr").copy()
    # cmap3.set_extremes(under='magenta', over='yellow')
    im3 = hep.hist2dplot((bias_arr_1 - bias_arr_2, bin_centers_i, bin_centers_j), ax=ax[1,0], cmap=cmap3)
    im3.pcolormesh.set_clim(-bias_amplitude, bias_amplitude)
    ax[1,0].set_title(title3, pad=tilte_pad)
    ax[1,0].set_xlabel(r"$\eta$", fontsize=ax_label_size)
    ax[1,0].set_ylabel(r"$\phi$", fontsize=ax_label_size)
    im3.cbar.remove()
    # im3.cbar.set_label(cbar_label_bias)
    # if extend_colorbars[2] != None:
    #     im3.cbar.remove()
    #     cax3 = hep.append_axes(ax3, size="7%", pad=0.2, position="right", extend=False)
    #     plt.colorbar(im3.pcolormesh, ax=ax[1,0], cax=cax3, extend=extend_colorbars[2], label=cbar_label_bias)

    bias_total_err_arr = np.sqrt(bias_err_arr_1**2 + bias_err_arr_2**2)

    cmap4 = plt.get_cmap("magma").copy()
    cmap4.set_extremes(under='magenta', over='lime')
    im4 = hep.hist2dplot((np.abs(bias_arr_1 - bias_arr_2) / bias_total_err_arr, bin_centers_i, bin_centers_j), ax=ax[1,1], cmap=cmap4)
    im4.pcolormesh.set_clim(0, bias_pull_amplitude)
    ax[1,1].set_title(title4, pad=tilte_pad)
    ax[1,1].set_xlabel(r"$\eta$", fontsize=ax_label_size)
    # ax[1,1].set_ylabel(r"$\phi$", fontsize=ax_label_size)
    im4.cbar.set_label(cbar_label_pull)
    if extend_colorbars[1] != None:
        im4.cbar.remove()
        cax4 = hep.append_axes(ax[1,1], size="7%", pad=0.2, position="right", extend=False)
        plt.colorbar(im4.pcolormesh, ax=ax[1,1], cax=cax4, extend=extend_colorbars[1], label=cbar_label_pull)

    plt.tight_layout()

    
    if save_fig:
        plt.savefig(save_path, transparent=False, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

In [ ]:
def get_fit(hist1, hist2, draw):
    # x = RooRealVar("x", "M^{+-}", 60, 160)
    # x = RooRealVar("x", "M^{+-}", 80, 100)
    x = RooRealVar("x", "M^{+-}", 70, 120)
    # x = RooRealVar("x", "M^{+-}", 60, 140)
    
    mean     = RooRealVar("Mean",   "", 80, 100)
    assym    = RooRealVar("A",   "", -1, 1)
    mu_1     = RooFormulaVar("mu_1", "", "Mean*(1+A)", RooArgList(mean, assym))
    mu_2     = RooFormulaVar("mu_2", "", "Mean*(1-A)", RooArgList(mean, assym))
    
    sigmaL_1 = RooRealVar("SigmaL1", "", 1, 30)
    sigmaR_1 = RooRealVar("SigmaR1", "", 1, 30)
    alphaL_1 = RooRealVar("AlphaL1", "", 0.5, 5)
    alphaR_1 = RooRealVar("AlphaR1", "", 0.5, 5)
    nL_1     = RooRealVar("NL1",    "",  0.1, 200)
    nR_1     = RooRealVar("NR1",    "",  0.1, 200)
    
    mean.setVal(90)
    assym.setVal(0)
    sigmaL_1.setVal(5)
    sigmaR_1.setVal(5)
    alphaL_1.setVal(1)
    alphaR_1.setVal(1)
    nL_1.setVal(1)
    nR_1.setVal(1)
    # nL_1.setVal(100)
    # nR_1.setVal(100)
    # nL_1.setConstant(True)
    # nR_1.setConstant(True)
    
    crystalball_1 = ROOT.RooCrystalBall("signal1", "", x, mu_1, sigmaL_1, sigmaR_1, alphaL_1, nL_1, alphaR_1, nR_1)
    crystalball_2 = ROOT.RooCrystalBall("signal2", "", x, mu_2, sigmaL_1, sigmaR_1, alphaL_1, nL_1, alphaR_1, nR_1)
    
    # norm_bg = RooRealVar("Bias", "", 0, 1)
    # norm_bg.setVal(0)
    # norm_bg.setConstant(True)
    # background = ROOT.RooPolynomial("background", "", x)
    
    category = ROOT.RooCategory("category", "")
    category.defineType("hist1")
    category.defineType("hist2")
    
    map1 = ROOT.std.map("std::string, TH1*")()
    map1.insert(("hist1", hist1))
    map1.insert(("hist2", hist2))
    
    combined_data = ROOT.RooDataHist("combined_data", "", RooArgList(x), category, map1)
    
    # model1 = ROOT.RooAddPdf("model1", "", background, crystalball_1, norm_bg)
    # model2 = ROOT.RooAddPdf("model2", "", background, crystalball_2, norm_bg)
    model1 = crystalball_1
    model2 = crystalball_2
    
    # Define a simultaneous model
    simultaneous_model = ROOT.RooSimultaneous("simultaneous_model", "", category)
    simultaneous_model.addPdf(model1, "hist1")
    simultaneous_model.addPdf(model2, "hist2")
    
    nll1 = simultaneous_model.createNLL(combined_data,
                                      # RooFit.Offset("initial"),
                                      )
    minimizer1 = ROOT.RooMinimizer(nll1)
    # minimizer1.setMinimizerType("GSLMultiMin")
    minimizer1.setMinimizerType("Minuit2")
    minimizer1.optimizeConst(True)
    # minimizer1.setStrategy(2)
    minimizer1.setEps(1e-8)
    minimizer1.migrad()
    minimizer1.improve()
    # minimizer1.hesse()
    # minimizer1.minos()
    result1 = minimizer1.save()

    # if nL_1.getVal() > 100:
    #     nL_1.setConstant(True)
    # if nR_1.getVal() > 100:
    #     nR_1.setConstant(True)
    
    nll2 = simultaneous_model.createNLL(combined_data,
                                      RooFit.Offset("initial"),
                                      # RooFit.IntegrateBins(1e-5),
                                      )
    minimizer2 = ROOT.RooMinimizer(nll2)
    minimizer2.setMinimizerType("Minuit2")
    minimizer2.optimizeConst(True)
    # minimizer2.setStrategy(2)
    minimizer2.setEps(1e-8)
    minimizer2.migrad()
    minimizer2.improve()
    minimizer2.hesse()
    # minimizer2.minos()
    result2 = minimizer2.save()

    all_roofit_objects = [x, mean, assym, mu_1, mu_2, sigmaL_1, sigmaR_1, alphaL_1, alphaR_1, nL_1, nR_1,
                          crystalball_1, crystalball_2, category, map1, combined_data,
                          simultaneous_model, nll1, nll2, minimizer1, minimizer2, result1, result2]

    params = [alphaL_1, alphaR_1, assym, mean, nL_1, nR_1, sigmaL_1, sigmaR_1]
    param_values = [param.getVal() for param in params]
    
    val_assym   = assym.getValV()
    error_assym = assym.getError()

    hist1_name  = f"hist1"
    hist2_name  = f"hist2"
    chi2_val = get_chi2_for_bin(hist1, hist2, model1, model2, x, hist1_name, hist2_name)
    
    if draw:
        canvas = ROOT.TCanvas(f"canvas_separate_{np.random.randint(0, 100000)}", "Separate Fit", 1200, 750)
        
        frame1 = x.frame(ROOT.RooFit.Title("Histogram 1"))
        combined_data.plotOn(frame1, ROOT.RooFit.Cut("category==category::hist1"), MarkerColor=ROOT.kRed, LineColor=ROOT.kRed)
        simultaneous_model.plotOn(frame1, ROOT.RooFit.Slice(category, "hist1"), ROOT.RooFit.ProjWData(category, combined_data), LineColor=ROOT.kRed)
        
        frame2 = x.frame(ROOT.RooFit.Title("Histogram 2"))
        combined_data.plotOn(frame2, ROOT.RooFit.Cut("category==category::hist2"), MarkerColor=ROOT.kBlue, LineColor=ROOT.kBlue)
        simultaneous_model.plotOn(frame2, ROOT.RooFit.Slice(category, "hist2"), ROOT.RooFit.ProjWData(category, combined_data), LineColor=ROOT.kBlue)
        frame2.Draw()
        frame1.Draw("Same")

        # all_roofit_objects.extend([canvas, frame1, frame2])

    SetOwnerships(all_roofit_objects)
    
    if draw:
        return val_assym, error_assym, param_values, chi2_val, canvas
    else:
        return val_assym, error_assym, param_values, chi2_val


In [ ]:
# h5_path = "output/_mz_dilepton_scetlib_dyturbo_CT18Z_N3p0LL_N2LO_Corr_mll.hdf5"
# h5_path = "output/mz_dilepton_scetlib_dyturbo_CT18Z_N3p0LL_N2LO_Corr_mll_2016PostVFP_wlike.hdf5"
h5_path = "output/mz_dilepton_scetlib_dyturbo_CT18Z_N3p0LL_N2LO_Corr_mll_2016PostVFP_dilepton.hdf5"
h5_file = h5py.File(h5_path, "r")
results = base_io.load_results_h5py(h5_file)
print(results.keys())

In [ ]:
groups = Datagroups(h5_path)
# print(groups.getNames())
# print(groups.getProcNames())
groups.loadHistsForDatagroups("nominal_etaPlus_pseudomass_phiPlus_pseudomass_pseudomassPlus", syst="")
groups.loadHistsForDatagroups("nominal_etaMinus_pseudomass_phiMinus_pseudomass_pseudomassMinus", syst="")
groups.loadHistsForDatagroups("nominal_etaPlus_pseudomass_phiPlus_pseudomass_ptPlus_reciprocal", syst="")
groups.loadHistsForDatagroups("nominal_etaMinus_pseudomass_phiMinus_pseudomass_ptMinus_reciprocal", syst="")

In [ ]:
groups.groups

In [ ]:
groups.groups["Zmumu"].members

In [ ]:
# groups.loadHistsForDatagroups('nominal_etaPlus_pseudomass_phiPlus_pseudomass_pseudomassPlus', '')
# groups.groups['Zmumu'].members[0].name
test_hists_flat = []

data_hist_Mplus = None
data_hist_Mminus = None
data_hist_Pplus_rec = None
data_hist_Pminus_rec = None


mc_hists_Mplus = []
mc_hists_Mminus = []
mc_hists_Pplus_rec = []
mc_hists_Pminus_rec = []

mc_hist_sum_Mplus = None
mc_hist_sum_Mminus = None
mc_hist_sum_Pplus_rec = None
mc_hist_sum_Pminus_rec = None

for group_name in groups.groups.keys():
    group = groups.groups[group_name]
    print(group.name)
    for group_member in group.members:
        print(group_member.name, group_member.is_data)
    print("=======")

    if group_name == "Data":
        data_hist_Mplus  = group.hists['nominal_etaPlus_pseudomass_phiPlus_pseudomass_pseudomassPlus']
        data_hist_Mminus = group.hists['nominal_etaMinus_pseudomass_phiMinus_pseudomass_pseudomassMinus']
        data_hist_Pplus_rec  = group.hists['nominal_etaPlus_pseudomass_phiPlus_pseudomass_ptPlus_reciprocal']
        data_hist_Pminus_rec = group.hists['nominal_etaMinus_pseudomass_phiMinus_pseudomass_ptMinus_reciprocal']
        
    else:
        mc_hists_Mplus .append(group.hists['nominal_etaPlus_pseudomass_phiPlus_pseudomass_pseudomassPlus'])
        mc_hists_Mminus.append(group.hists['nominal_etaMinus_pseudomass_phiMinus_pseudomass_pseudomassMinus'])
        mc_hists_Pplus_rec .append(group.hists['nominal_etaPlus_pseudomass_phiPlus_pseudomass_ptPlus_reciprocal'])
        mc_hists_Pminus_rec.append(group.hists['nominal_etaMinus_pseudomass_phiMinus_pseudomass_ptMinus_reciprocal'])

        if mc_hist_sum_Mplus is not None:
            mc_hist_sum_Mplus  = hh.addHists(mc_hist_sum_Mplus , group.hists['nominal_etaPlus_pseudomass_phiPlus_pseudomass_pseudomassPlus'])
            mc_hist_sum_Mminus = hh.addHists(mc_hist_sum_Mminus, group.hists['nominal_etaMinus_pseudomass_phiMinus_pseudomass_pseudomassMinus'])
            mc_hist_Pplus_rec  = hh.addHists(mc_hist_sum_Pplus_rec , group.hists['nominal_etaPlus_pseudomass_phiPlus_pseudomass_ptPlus_reciprocal'])
            mc_hist_Pminus_rec = hh.addHists(mc_hist_sum_Pminus_rec, group.hists['nominal_etaMinus_pseudomass_phiMinus_pseudomass_ptMinus_reciprocal'])
        else:
            mc_hist_sum_Mplus      = group.hists['nominal_etaPlus_pseudomass_phiPlus_pseudomass_pseudomassPlus']
            mc_hist_sum_Mminus     = group.hists['nominal_etaMinus_pseudomass_phiMinus_pseudomass_pseudomassMinus']
            mc_hist_sum_Pplus_rec  = group.hists['nominal_etaPlus_pseudomass_phiPlus_pseudomass_ptPlus_reciprocal']
            mc_hist_sum_Pminus_rec = group.hists['nominal_etaMinus_pseudomass_phiMinus_pseudomass_ptMinus_reciprocal']
    
    test_hists_flat.append(group.hists['nominal_etaPlus_pseudomass_phiPlus_pseudomass_pseudomassPlus'].integrate("phiPlus_pseudomass").integrate("etaPlus_pseudomass"))

# data_hist_Mplus        = data_hist_Mplus       [::1j,::2j,:]
# data_hist_Mminus       = data_hist_Mminus      [::1j,::2j,:]
# data_hist_Pplus_rec    = data_hist_Pplus_rec   [::1j,::2j,:]
# data_hist_Pminus_rec   = data_hist_Pminus_rec  [::1j,::2j,:]
# mc_hist_sum_Mplus      = mc_hist_sum_Mplus     [::1j,::2j,:]
# mc_hist_sum_Mminus     = mc_hist_sum_Mminus    [::1j,::2j,:]
# mc_hist_sum_Pplus_rec  = mc_hist_sum_Pplus_rec [::1j,::2j,:]
# mc_hist_sum_Pminus_rec = mc_hist_sum_Pminus_rec[::1j,::2j,:]

# data_hist_Mplus        = data_hist_Mplus       [::2j,::2j,:]
# data_hist_Mminus       = data_hist_Mminus      [::2j,::2j,:]
# data_hist_Pplus_rec    = data_hist_Pplus_rec   [::2j,::2j,:]
# data_hist_Pminus_rec   = data_hist_Pminus_rec  [::2j,::2j,:]
# mc_hist_sum_Mplus      = mc_hist_sum_Mplus     [::2j,::2j,:]
# mc_hist_sum_Mminus     = mc_hist_sum_Mminus    [::2j,::2j,:]
# mc_hist_sum_Pplus_rec  = mc_hist_sum_Pplus_rec [::2j,::2j,:]
# mc_hist_sum_Pminus_rec = mc_hist_sum_Pminus_rec[::2j,::2j,:]

data_hist_Mplus        = data_hist_Mplus       [::4j,::4j,:]
data_hist_Mminus       = data_hist_Mminus      [::4j,::4j,:]
data_hist_Pplus_rec    = data_hist_Pplus_rec   [::4j,::4j,:]
data_hist_Pminus_rec   = data_hist_Pminus_rec  [::4j,::4j,:]
mc_hist_sum_Mplus      = mc_hist_sum_Mplus     [::4j,::4j,:]
mc_hist_sum_Mminus     = mc_hist_sum_Mminus    [::4j,::4j,:]
mc_hist_sum_Pplus_rec  = mc_hist_sum_Pplus_rec [::4j,::4j,:]
mc_hist_sum_Pminus_rec = mc_hist_sum_Pminus_rec[::4j,::4j,:]

# data_hist_Mplus        = data_hist_Mplus       [::8j,::10j,:]
# data_hist_Mminus       = data_hist_Mminus      [::8j,::10j,:]
# data_hist_Pplus_rec    = data_hist_Pplus_rec   [::8j,::10j,:]
# data_hist_Pminus_rec   = data_hist_Pminus_rec  [::8j,::10j,:]
# mc_hist_sum_Mplus      = mc_hist_sum_Mplus     [::8j,::10j,:]
# mc_hist_sum_Mminus     = mc_hist_sum_Mminus    [::8j,::10j,:]
# mc_hist_sum_Pplus_rec  = mc_hist_sum_Pplus_rec [::8j,::10j,:]
# mc_hist_sum_Pminus_rec = mc_hist_sum_Pminus_rec[::8j,::10j,:]

# data_hist_Mplus        = data_hist_Mplus       [::20j,::10j,:]
# data_hist_Mminus       = data_hist_Mminus      [::20j,::10j,:]
# data_hist_Pplus_rec    = data_hist_Pplus_rec   [::20j,::10j,:]
# data_hist_Pminus_rec   = data_hist_Pminus_rec  [::20j,::10j,:]
# mc_hist_sum_Mplus      = mc_hist_sum_Mplus     [::20j,::10j,:]
# mc_hist_sum_Mminus     = mc_hist_sum_Mminus    [::20j,::10j,:]
# mc_hist_sum_Pplus_rec  = mc_hist_sum_Pplus_rec [::20j,::10j,:]
# mc_hist_sum_Pminus_rec = mc_hist_sum_Pminus_rec[::20j,::10j,:]

# data_hist_Mplus        = data_hist_Mplus       [::4j,::5j,:]
# data_hist_Mminus       = data_hist_Mminus      [::4j,::5j,:]
# data_hist_Pplus_rec    = data_hist_Pplus_rec   [::4j,::5j,:]
# data_hist_Pminus_rec   = data_hist_Pminus_rec  [::4j,::5j,:]
# mc_hist_sum_Mplus      = mc_hist_sum_Mplus     [::4j,::5j,:]
# mc_hist_sum_Mminus     = mc_hist_sum_Mminus    [::4j,::5j,:]
# mc_hist_sum_Pplus_rec  = mc_hist_sum_Pplus_rec [::4j,::5j,:]
# mc_hist_sum_Pminus_rec = mc_hist_sum_Pminus_rec[::4j,::5j,:]

# data_hist_Mplus        = data_hist_Mplus       [::40j,::20j,:]
# data_hist_Mminus       = data_hist_Mminus      [::40j,::20j,:]
# data_hist_Pplus_rec    = data_hist_Pplus_rec   [::40j,::20j,:]
# data_hist_Pminus_rec   = data_hist_Pminus_rec  [::40j,::20j,:]
# mc_hist_sum_Mplus      = mc_hist_sum_Mplus     [::40j,::20j,:]
# mc_hist_sum_Mminus     = mc_hist_sum_Mminus    [::40j,::20j,:]
# mc_hist_sum_Pplus_rec  = mc_hist_sum_Pplus_rec [::40j,::20j,:]
# mc_hist_sum_Pminus_rec = mc_hist_sum_Pminus_rec[::40j,::20j,:]

In [ ]:
group.hists['nominal_etaPlus_pseudomass_phiPlus_pseudomass_pseudomassPlus']

In [ ]:
data_hist_Mplus

In [ ]:
egg = data_hist_Pplus_rec[0,0,::]
egg

In [ ]:
@interact(i=(0, data_hist_Mplus.axes[0].size-1), j=(0,data_hist_Mplus.axes[1].size-1))
def _plot(i,j):
    # return data_hist_Mplus[i,j,:]
    return data_hist_Pplus_rec[i,j,:]

In [ ]:
pseudomass_external = np.array([ 17885.,  18202.,  18878.,  20017.,  20669.,  21710.,  22378.,
        23534.,  24488.,  25254.,  26309.,  26879.,  28018.,  29294.,
        30787.,  31820.,  33004.,  34095.,  35719.,  37080.,  38936.,
        39868.,  41457.,  43624.,  44893.,  46899.,  48609.,  51241.,
        52974.,  54762.,  57053.,  59540.,  62179.,  65086.,  67399.,
        70994.,  73315.,  76759.,  80711.,  83643.,  87843.,  91613.,
        95005.,  99875., 104407., 109275., 113966., 119088., 124366.,
       130092., 135134., 141145., 146667., 152184., 156891., 162375.,
       166554., 170512., 172648., 176110., 177120., 176196., 176211.,
       173892., 172534., 167853., 162194., 157666., 151666., 146169.,
       140317., 133832., 128287., 122188., 115631., 110325., 105385.,
        99972.,  95512.,  91342.,  86439.,  81961.,  78369.,  74228.,
        71145.,  67370.,  65172.,  61588.,  59107.,  56285.,  53823.,
        51239.,  49377.,  47471.,  45626.,  44153.,  41770.,  40047.,
        38516.,  37043.,  35541.,  34130.,  32954.,  31641.,  30513.,
        29393.,  28336.,  27322.,  26608.,  25442.,  24811.,  23506.,
        23101.,  22213.,  21434.,  20806.,  20014.,  19257.,  18608.,
        18116.,  17755.,  17080.,  16628.,  16157.,  15533.,  15099.,
        14691.,  13947.,  13895.,  13272.,  12869.,  12546.,  12109.,
        11653.,  11396.,  11220.,  10713.,  10421.,  10054.,  10031.,
         9738.,   9363.,   9154.,   9009.,   8742.,   8478.,   8090.,
         7861.,   7607.,   7524.,   7323.,   7096.,   6993.,   6788.,
         6630.,   6312.,   6172.,   6154.,   5883.,   5632.,   5582.,
         5484.,   5304.,   5247.,   5102.,   4799.,   4770.,   4689.,
         4666.,   4388.,   4378.,   4143.,   4108.,   3994.,   3941.,
         3809.,   3659.,   3643.,   3578.,   3493.,   3371.,   3241.,
         3114.,   3015.,   3088.,   2978.,   2835.,   2756.,   2634.,
         2666.,   2556.,   2531.,   2359.,   2338.,   2206.,   2204.,
         2100.,   2122.,   1997.,   1981.])

In [ ]:
# fig, ax = plt.subplots()
print(test_hists_flat[0].sum().value)
print(np.sum(pseudomass_external))
print(test_hists_flat[0].sum().value / np.sum(pseudomass_external))

test_hists_flat[0].plot(label="Wremnants Data")
sum(test_hists_flat[1:]).plot(label="Wremnants MC")
plt.step(np.linspace(60, 160, 200), pseudomass_external, label="my Data")
plt.yscale("log")
plt.ylim(1e3,2e5)
plt.xlim(60,140)
plt.grid()
plt.legend()

In [ ]:
i, j = 0, 0
test_centers = data_hist_Pplus_rec [i,j,:].axes[0].centers
test_weights = data_hist_Pplus_rec [i,j,:].values()

def get_bootstrapped_std_error(centers, weights):
    test_data = np.stack([centers, weights], axis=0)
    def test_fun(*arr, axis):
        arr = np.array(arr)
        centers = arr[0,:]
        weights = arr[1,:]
        out = np.average(centers, weights=weights, axis=axis)
        return out

    result = stats.bootstrap(test_data, test_fun, vectorized=True, axis=0, paired=True)
    return result.standard_error

get_bootstrapped_std_error(test_centers, test_weights)

In [ ]:
np.average(test_centers, weights=test_weights)

In [ ]:
# egg = data_hist_Mplus[0,0,:]
# egg_root = make_root_hist(egg)


In [ ]:
if 'canvas' not in globals():
    canvas = ROOT.TCanvas("canvas", "", 1000, 600)

test_hist_1 = make_root_hist(data_hist_Mplus[0,0,:])
test_hist_2 = make_root_hist(data_hist_Mminus[0,0,:])

test_hist_1.SetStats(0)

# test_hist_1.Scale(1/test_hist_1.GetSum())
# test_hist_2.Scale(1/test_hist_2.GetSum())

test_hist_1.SetLineColor(kRed)

test_hist_1.Draw()
test_hist_2.Draw("Same")

canvas.Draw()

In [ ]:
def get_bootstrapped_std_error(centers, weights):
    test_data = np.stack([centers, weights], axis=0)
    def test_fun(*arr, axis):
        arr = np.array(arr)
        centers = arr[0,:]
        weights = arr[1,:]
        out = np.average(centers, weights=weights, axis=axis)
        return out

    result = stats.bootstrap(test_data, test_fun, vectorized=True, axis=0, paired=True, method="BCa")
    return result.standard_error

def compute_fit_arrays(hist_Mplus, hist_Mminus, hist_Pplus_rec, hist_Pminus_rec, make_plots=False):
    shape_hist = hist_Mplus.shape[:2]
    
    muPlus_pmag_reciprocal_mean_arr  = np.zeros(shape_hist)
    muMinus_pmag_reciprocal_mean_arr = np.zeros(shape_hist)
    Pplus_rec_error_arr              = np.zeros(shape_hist)
    Pminus_rec_error_arr             = np.zeros(shape_hist)
    assym_arr_separate_fit           = np.zeros(shape_hist)
    assym_error_arr_separate_fit     = np.zeros(shape_hist)
    # bias_arr_separate_fit            = np.zeros(shape_hist)
    # bias_error_arr_separate_fit      = np.zeros(shape_hist)
    chi2_arr_separate_fit            = np.zeros(shape_hist)
    
    curve_param_arr_separate_fit = np.zeros([*shape_hist, 6]) # sigmaL, sigmaR, alphaL, alphaR, nL, nR
    
    hist_12_ij_list_list_separate = []
    
    # plot_i = 9
    # plot_j = 4
    
    # plot_done = False
    # make_plots = True
    
    # plot_i = -999
    # plot_j = -999

    if make_plots:
        canvas_arr = np.empty(shape_hist, dtype=object)
    
    for i in tqdm(range(0, shape_hist[0])):
        param_values_i = []
        hist_12_ij_list_separate = []
        for j in range(0, shape_hist[1]):
            # if plot_done:
            #     break
            subfit_label = f"{i}_{j}"
            
            hist1_ij = make_root_hist(hist_Mplus [i,j,:])
            hist2_ij = make_root_hist(hist_Mminus[i,j,:])
    
            hist1_ij.SetName(f"hist1_{subfit_label}")
            hist2_ij.SetName(f"hist2_{subfit_label}")
            
            # if i == plot_i and j == plot_j:
            if make_plots:
                assym_current, assym_current_err, param_values_current, chi2_val, canvas_current = get_fit(hist1_ij, hist2_ij, draw=True)
                canvas_arr[i,j] = canvas_current
                # plot_done = True
            else:
                assym_current, assym_current_err, param_values_current, chi2_val = get_fit(hist1_ij, hist2_ij, draw=False)
            # SetOwnerships([hist1_ij, hist2_ij])
    
            assym_arr_separate_fit[i,j]       = assym_current
            assym_error_arr_separate_fit[i,j] = assym_current_err
            # alphaL_arr_separate_fit[i,j] = param_values_current[0]
            # alphaR_arr_separate_fit[i,j] = param_values_current[1]
            # nL_arr_separate_fit    [i,j] = param_values_current[4]
            # nR_arr_separate_fit    [i,j] = param_values_current[5]
            # sigmaL_arr_separate_fit[i,j] = param_values_current[6]
            # sigmaR_arr_separate_fit[i,j] = param_values_current[7]
            curve_param_arr_separate_fit[i,j,0] = param_values_current[0]
            curve_param_arr_separate_fit[i,j,1] = param_values_current[1]
            curve_param_arr_separate_fit[i,j,2] = param_values_current[4]
            curve_param_arr_separate_fit[i,j,3] = param_values_current[5]
            curve_param_arr_separate_fit[i,j,4] = param_values_current[6]
            curve_param_arr_separate_fit[i,j,5] = param_values_current[7]
    
            chi2_arr_separate_fit[i,j] = chi2_val

            # np.average(egg.axes[0].centers, weights=egg.values())
            # muPlus_pmag_reciprocal_mean_arr [i,j] = np.mean(hist_Pplus_rec [i,j,:].values())
            # muMinus_pmag_reciprocal_mean_arr[i,j] = np.mean(hist_Pminus_rec[i,j,:].values())
            muPlus_pmag_reciprocal_mean_arr [i,j] = np.average(hist_Pplus_rec [i,j,:].axes[0].centers, weights=hist_Pplus_rec [i,j,:].values())
            muMinus_pmag_reciprocal_mean_arr[i,j] = np.average(hist_Pminus_rec[i,j,:].axes[0].centers, weights=hist_Pminus_rec[i,j,:].values())

            Pplus_rec_error_arr [i,j] = get_bootstrapped_std_error(hist_Pplus_rec [i,j,:].axes[0].centers, weights=hist_Pplus_rec [i,j,:].values())
            Pminus_rec_error_arr[i,j] = get_bootstrapped_std_error(hist_Pminus_rec[i,j,:].axes[0].centers, weights=hist_Pminus_rec[i,j,:].values())
    
            # hist_12_ij_list_separate.append([hist1_ij, hist2_ij])
        # hist_12_ij_list_list_separate.append(hist_12_ij_list_separate)
    
    bias_arr_separate_fit         = -assym_arr_separate_fit * 1/2 * (muPlus_pmag_reciprocal_mean_arr + muMinus_pmag_reciprocal_mean_arr)
    bias_error_arr_separate_fit_1 = assym_error_arr_separate_fit * 1/2 * (muPlus_pmag_reciprocal_mean_arr + muMinus_pmag_reciprocal_mean_arr)
    bias_error_arr_separate_fit_2 = np.abs(assym_arr_separate_fit * 1/2 * Pplus_rec_error_arr)
    bias_error_arr_separate_fit_3 = np.abs(assym_arr_separate_fit * 1/2 * Pminus_rec_error_arr)
    # Correlations are expected to be negligible
    bias_error_arr_separate_fit   = (bias_error_arr_separate_fit_1**2 + bias_error_arr_separate_fit_2**2 + bias_error_arr_separate_fit_3**2)**0.5

    if make_plots:
        return bias_arr_separate_fit, bias_error_arr_separate_fit, chi2_arr_separate_fit, curve_param_arr_separate_fit, canvas_arr
    else:
        return bias_arr_separate_fit, bias_error_arr_separate_fit, chi2_arr_separate_fit, curve_param_arr_separate_fit

# data_hist_Mplus     
# data_hist_Mminus    
# data_hist_Pplus_rec 
# data_hist_Pminus_rec
# mc_hist_sum_Mplus     
# mc_hist_sum_Mminus    
# mc_hist_sum_Pplus_rec 
# mc_hist_sum_Pminus_rec

# bias_arr_data, bias_error_arr_data, chi2_arr_data, curve_param_arr_data = compute_fit_arrays(data_hist_Mplus, data_hist_Mminus, data_hist_Pplus_rec, data_hist_Pminus_rec)
# bias_arr_mc_sum, bias_error_arr_mc_sum, chi2_arr_mc_sum, curve_param_arr_mc_sum = compute_fit_arrays(mc_hist_sum_Mplus, mc_hist_sum_Mminus, mc_hist_sum_Pplus_rec, mc_hist_sum_Pminus_rec)

bias_arr_data, bias_error_arr_data, chi2_arr_data, curve_param_arr_data, canvas_arr_data       = compute_fit_arrays(data_hist_Mplus  , data_hist_Mminus  , data_hist_Pplus_rec  , data_hist_Pminus_rec  , make_plots=True)
bias_arr_mc_sum, bias_error_arr_mc_sum, chi2_arr_mc_sum, curve_param_arr_mc_sum, canvas_arr_mc = compute_fit_arrays(mc_hist_sum_Mplus, mc_hist_sum_Mminus, mc_hist_sum_Pplus_rec, mc_hist_sum_Pminus_rec, make_plots=True)

In [ ]:
if 'canvas_arr_data' in globals():
    canvas_arr_data[0,0].Draw()

In [ ]:
canvas_arr_data[0,0].Draw()

In [ ]:
canvas_arr_mc[0,0].Draw()

In [ ]:
egg = make_root_hist(data_hist_Mplus[0,0,:])

In [ ]:
# plot fit variables

def plot_arr(arr_to_plot, bin_centers_i, bin_centers_j, title, cmap="viridis", cmin=None, cmax=None):
    fig, ax = plt.subplots(1,1, figsize=(11,6), dpi=100)
    hep.cms.label(loc=0, rlabel="", data=True, label="Preliminary", ax=ax)
    im = hep.hist2dplot(arr_to_plot, bin_centers_i, bin_centers_j, ax=ax, cmap=cmap, cmin=cmin, cmax=cmax)
    ax.set_title(title, pad=20)
    ax.set_xlabel(r"$\eta$")
    ax.set_ylabel(r"$\phi$")
    plt.tight_layout()
    plt.show()


bin_extents_eta = data_hist_Mplus.axes[0].edges
bin_extents_phi = data_hist_Mplus.axes[1].edges
ndof = 200*2-8

plot_arr(bias_arr_data, bin_extents_eta, bin_extents_phi, "bias" ,cmap="bwr")
plot_arr(curve_param_arr_data[:,:,0], bin_extents_eta, bin_extents_phi, "alpha_L")
plot_arr(curve_param_arr_data[:,:,1], bin_extents_eta, bin_extents_phi, "alpha_R")
plot_arr(curve_param_arr_data[:,:,2], bin_extents_eta, bin_extents_phi, "n_L")
plot_arr(curve_param_arr_data[:,:,3], bin_extents_eta, bin_extents_phi, "n_R")
plot_arr(curve_param_arr_data[:,:,4], bin_extents_eta, bin_extents_phi, "sigma_L")
plot_arr(curve_param_arr_data[:,:,5], bin_extents_eta, bin_extents_phi, "sigma_R")
plot_arr(chi2_arr_data / ndof, bin_extents_eta, bin_extents_phi, "chi2/ndof"
         # , cmin=0, cmax=1000
)

print("Mean chi2/ndof:", np.mean(chi2_arr_data) / ndof)
# print("Chi2/ndof for plot:", chi2_arr_data[plot_i, plot_j] / ndof)

In [ ]:
plot_bias_distributions(bias_arr_data, bias_error_arr_data, data_hist_Mplus.axes[0].edges, data_hist_Mplus.axes[1].edges, is_data=True, save_fig=False,
                        bias_amplitude=None, bias_err_amplitude=None, extend_colorbars=[None, None, None])

In [ ]:
plot_bias_distributions(bias_arr_mc_sum, bias_error_arr_mc_sum, data_hist_Mplus.axes[0].edges, data_hist_Mplus.axes[1].edges, is_data=True, save_fig=False,
                        bias_amplitude=None, bias_err_amplitude=None, extend_colorbars=[None, None, None])

In [ ]:
title = "2016_postVFP"

plot_bias_comparison(bias_arr_data, bias_arr_mc_sum, data_hist_Mplus.axes[0].edges, data_hist_Mplus.axes[1].edges,
                     bias_error_arr_data, bias_error_arr_mc_sum,
                     title=title,
                     title1="Data", title2="MC", title3="Difference", title4="Pull of difference",
                     cbar_label_bias="1/GeV",
                     save_fig=False, save_path=None,
                     bias_amplitude=None, bias_pull_amplitude=None, extend_colorbars=[None, None], )

In [ ]:
bias_difference       = bias_arr_data - bias_arr_mc_sum
bias_difference_error = (bias_error_arr_data**2 + bias_error_arr_mc_sum**2)**0.5

mean_bias_difference       = np.mean(bias_difference)
mean_bias_difference_error = np.sum(bias_difference_error**2)**0.5 / bias_difference_error.size

print(f"Mean bias difference: {mean_bias_difference:.04g} +- {mean_bias_difference_error:.04g} 1/GeV")

In [ ]:
mean_bias_difference / mean_bias_difference_error

In [ ]:
bias_weights = data_hist_Mplus[:,:,::hist.sum].to_numpy()[0] + data_hist_Mminus[:,:,::hist.sum].to_numpy()[0]

bias_difference       = bias_arr_data - bias_arr_mc_sum
bias_difference_error = (bias_error_arr_data**2 + bias_error_arr_mc_sum**2)**0.5

mean_bias_difference       = np.average(bias_difference, weights=bias_weights)
mean_bias_difference_error = np.sum(bias_weights**2 * bias_difference_error**2)**0.5 / np.sum(bias_weights)

print(f"Weighted mean of the bias difference: {mean_bias_difference:.04g} +- {mean_bias_difference_error:.04g} 1/GeV")

In [ ]:
data_hist_Mplus[:,:,::hist.sum].to_numpy()[0]